# AI in Fluids - Week 3 Lab 1
## Maxwellian sampling, macroscopic moments, DSMC noise, and noisy ML labels

**Course story:** Week 1 represented a fluid by grid fields. Week 2 trained a neural-network surrogate. This notebook shows where kinetic/DSMC field data come from and why finite particle sampling creates an error floor for machine learning.

### Submission reminder
Submit the **completed notebook with visible outputs** and include all requested interpretation in the single **Week-3 PDF report**. The notebook alone is not a complete submission.

## Learning objectives
By the end of this notebook you should be able to:

1. sample molecular velocities from a Maxwellian distribution;
2. estimate bulk velocity, temperature, pressure, and heat flux from particles;
3. demonstrate that sampling error decreases approximately as the inverse square root of the number of independent samples;
4. explain why a neural network trained on noisy DSMC labels learns the labels, not an inaccessible exact solution.

## 0. Imports and reproducibility

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error

np.set_printoptions(precision=5, suppress=True)
SEED = 12
rng = np.random.default_rng(SEED)
print("Random seed:", SEED)

## 1. Maxwellian molecular velocities
For a monatomic equilibrium gas, each Cartesian molecular-velocity component is normally distributed. The component standard deviation is set by temperature and molecular mass. A bulk velocity shifts the center of the distribution but does not represent the thermal fluctuation itself.

In [ ]:
K_B = 1.380649e-23                    # Boltzmann constant [J/K]
M_ARGON = 6.6335209e-26              # argon molecular mass [kg]


def sample_maxwellian(n_particles, temperature, bulk_velocity=(0.0, 0.0, 0.0), seed=None):
    """Sample 3-D molecular velocities from an equilibrium Maxwellian."""
    local_rng = np.random.default_rng(seed)
    bulk_velocity = np.asarray(bulk_velocity, dtype=float)
    sigma = np.sqrt(K_B * temperature / M_ARGON)
    return bulk_velocity + sigma * local_rng.standard_normal((n_particles, 3))


T_TRUE = 300.0                         # K
U_TRUE = np.array([100.0, 0.0, 0.0])  # m/s
N_SAMPLE = 5000

c = sample_maxwellian(N_SAMPLE, T_TRUE, U_TRUE, seed=SEED)
sigma_true = np.sqrt(K_B * T_TRUE / M_ARGON)
print(f"Thermal component standard deviation = {sigma_true:.2f} m/s")
print("Molecular velocity array shape:", c.shape)

In [ ]:
# Plot one velocity component and the thermal-speed distribution.
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

bins = np.linspace(U_TRUE[0] - 4*sigma_true, U_TRUE[0] + 4*sigma_true, 55)
axes[0].hist(c[:, 0], bins=bins, density=True, alpha=0.65, label="samples")
x = np.linspace(bins[0], bins[-1], 400)
gaussian = np.exp(-0.5*((x-U_TRUE[0])/sigma_true)**2)/(sigma_true*np.sqrt(2*np.pi))
axes[0].plot(x, gaussian, lw=2, label="component PDF")
axes[0].axvline(U_TRUE[0], ls="--", lw=1.5, label="bulk velocity")
axes[0].set_xlabel(r"$c_x$ [m/s]")
axes[0].set_ylabel("probability density")
axes[0].set_title("Molecular component")
axes[0].legend()
axes[0].grid(alpha=0.2)

thermal_speed = np.linalg.norm(c - U_TRUE, axis=1)
bins_s = np.linspace(0, np.percentile(thermal_speed, 99.8), 55)
axes[1].hist(thermal_speed, bins=bins_s, density=True, alpha=0.65, label="samples")
s = np.linspace(0, bins_s[-1], 400)
maxwell_pdf = 4*np.pi*s**2*(M_ARGON/(2*np.pi*K_B*T_TRUE))**1.5 * np.exp(-M_ARGON*s**2/(2*K_B*T_TRUE))
axes[1].plot(s, maxwell_pdf, lw=2, label="Maxwell speed PDF")
axes[1].set_xlabel(r"$|\mathbf{c}-\mathbf{U}|$ [m/s]")
axes[1].set_ylabel("probability density")
axes[1].set_title("Thermal speed")
axes[1].legend()
axes[1].grid(alpha=0.2)

plt.tight_layout()
plt.show()

### Report prompt 1
Explain why the distribution width is much larger than the imposed bulk speed in this example. Distinguish molecular velocity from bulk velocity.

> **Your response:** Write a concise answer here for your own record, and include the final version in the PDF report.
>
> ...

## 2. Macroscopic quantities as particle moments

In [ ]:
def estimate_macroscopic_quantities(velocities, number_density, molecular_mass=M_ARGON):
    """Estimate monatomic-gas moments from a finite molecular sample."""
    U_hat = velocities.mean(axis=0)
    fluct = velocities - U_hat
    fluct2 = np.sum(fluct**2, axis=1)
    T_hat = molecular_mass * fluct2.mean() / (3.0 * K_B)
    p_hat = number_density * K_B * T_hat
    q_hat = 0.5 * molecular_mass * number_density * np.mean(fluct2[:, None] * fluct, axis=0)
    return U_hat, T_hat, p_hat, q_hat


N0 = 2.5e25  # illustrative argon number density [1/m^3]
U_hat, T_hat, p_hat, q_hat = estimate_macroscopic_quantities(c, N0)
q_scale = N0 * K_B * T_TRUE * sigma_true

summary = pd.DataFrame({
    "quantity": ["Ux [m/s]", "Uy [m/s]", "Uz [m/s]", "T [K]", "p [Pa]"],
    "imposed": [U_TRUE[0], U_TRUE[1], U_TRUE[2], T_TRUE, N0*K_B*T_TRUE],
    "estimated": [U_hat[0], U_hat[1], U_hat[2], T_hat, p_hat]
})
display(summary)
print("Estimated heat-flux vector [W/m^2]:", q_hat)
print("Normalized heat flux q/(n k_B T sigma):", q_hat/q_scale)

### Report prompt 2
Why is the bulk velocity subtracted before calculating temperature? Why is the sampled equilibrium heat flux not exactly zero?

> **Your response:** Write a concise answer here for your own record, and include the final version in the PDF report.
>
> ...

## 3. Statistical noise versus number of particles

In [ ]:
def repeated_sampling_errors(n_particles, repeats=200, seed=1):
    local_rng = np.random.default_rng(seed)
    u_errors = []
    t_errors = []
    qstar_norms = []
    for _ in range(repeats):
        velocities = U_TRUE + sigma_true * local_rng.standard_normal((n_particles, 3))
        Ue, Te, _, qe = estimate_macroscopic_quantities(velocities, N0)
        u_errors.append(Ue[0] - U_TRUE[0])
        t_errors.append(Te - T_TRUE)
        qstar_norms.append(np.linalg.norm(qe/q_scale))
    return {
        "rmse_Ux": np.sqrt(np.mean(np.square(u_errors))),
        "rmse_T": np.sqrt(np.mean(np.square(t_errors))),
        "mean_qstar_norm": np.mean(qstar_norms)
    }


N_VALUES = np.array([30, 60, 120, 300, 600, 1500, 3000, 8000])
rows = []
for n in N_VALUES:
    result = repeated_sampling_errors(int(n), repeats=220, seed=100+n)
    rows.append({"N": n, **result})
noise_df = pd.DataFrame(rows)
display(noise_df)

slope_u = np.polyfit(np.log10(noise_df["N"]), np.log10(noise_df["rmse_Ux"]), 1)[0]
slope_t = np.polyfit(np.log10(noise_df["N"]), np.log10(noise_df["rmse_T"]), 1)[0]
print(f"Fitted Ux RMSE slope = {slope_u:.3f}")
print(f"Fitted T  RMSE slope = {slope_t:.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))

axes[0].loglog(noise_df["N"], noise_df["rmse_Ux"], "o-", label=f"Ux RMSE, slope={slope_u:.2f}")
axes[0].loglog(noise_df["N"], noise_df["rmse_T"], "s-", label=f"T RMSE, slope={slope_t:.2f}")
reference = noise_df["rmse_T"].iloc[0] * (noise_df["N"]/noise_df["N"].iloc[0])**(-0.5)
axes[0].loglog(noise_df["N"], reference, "--", label=r"$N^{-1/2}$ reference")
axes[0].set_xlabel("independent particle samples, N")
axes[0].set_ylabel("RMSE")
axes[0].set_title("Sampling convergence")
axes[0].grid(which="both", alpha=0.25)
axes[0].legend()

axes[1].loglog(noise_df["N"], noise_df["mean_qstar_norm"], "o-")
axes[1].set_xlabel("independent particle samples, N")
axes[1].set_ylabel(r"mean $|q|/(n k_B T \sigma_c)$")
axes[1].set_title("Equilibrium heat-flux noise")
axes[1].grid(which="both", alpha=0.25)

plt.tight_layout()
plt.show()

### Report prompt 3
Report both fitted slopes. If the effective number of independent samples increases by a factor of 100, predict the approximate error reduction. Explain why time-correlated DSMC samples may have fewer independent samples than the raw sample count suggests.

> **Your response:** Write a concise answer here for your own record, and include the final version in the PDF report.
>
> ...

## 4. ML bridge: the network learns the labels it receives

In [ ]:
def hidden_slip_trend(log10_kn):
    """A smooth teaching target used only to expose the effect of noisy labels."""
    kn = 10.0**np.asarray(log10_kn)
    return 0.04 + 0.82*kn/(kn + 0.08)


log_kn_train = np.linspace(-3.0, 1.0, 75)
y_truth_train = hidden_slip_trend(log_kn_train)
label_rng = np.random.default_rng(18)

# Short averaging produces noisier labels; long averaging produces cleaner labels.
noise_short = 0.075*(1.0 + 0.25*(log_kn_train+3.0)/4.0)
noise_long = 0.018*(1.0 + 0.25*(log_kn_train+3.0)/4.0)
y_short = np.clip(y_truth_train + label_rng.normal(0.0, noise_short), 0.0, None)
y_long = np.clip(y_truth_train + label_rng.normal(0.0, noise_long), 0.0, None)


def make_small_mlp(random_state=4):
    return make_pipeline(
        StandardScaler(),
        MLPRegressor(hidden_layer_sizes=(24, 24), activation="tanh", solver="lbfgs",
                     alpha=1e-3, max_iter=3000, random_state=random_state)
    )

model_short = make_small_mlp()
model_long = make_small_mlp()
model_short.fit(log_kn_train[:, None], y_short)
model_long.fit(log_kn_train[:, None], y_long)

log_kn_test = np.linspace(-3.0, 1.0, 400)
y_truth_test = hidden_slip_trend(log_kn_test)
pred_short = model_short.predict(log_kn_test[:, None])
pred_long = model_long.predict(log_kn_test[:, None])

rmse_short = np.sqrt(mean_squared_error(y_truth_test, pred_short))
rmse_long = np.sqrt(mean_squared_error(y_truth_test, pred_long))
print(f"RMSE vs hidden truth, short averaging labels: {rmse_short:.4f}")
print(f"RMSE vs hidden truth, long  averaging labels: {rmse_long:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), sharey=True)
for ax, labels, prediction, title, rmse in [
    (axes[0], y_short, pred_short, "Short averaging / noisy labels", rmse_short),
    (axes[1], y_long, pred_long, "Long averaging / cleaner labels", rmse_long),
]:
    ax.scatter(log_kn_train, labels, s=20, alpha=0.65, label="training labels")
    ax.plot(log_kn_test, y_truth_test, lw=2.2, label="hidden smooth trend")
    ax.plot(log_kn_test, prediction, lw=2, label="MLP prediction")
    ax.set_xlabel(r"$\log_{10}(Kn)$")
    ax.set_title(f"{title}\nRMSE vs truth = {rmse:.3f}")
    ax.grid(alpha=0.2)
axes[0].set_ylabel("normalized slip proxy")
axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
# Basic physical checks on the two learned curves.
def monotonicity_violations(y):
    return int(np.sum(np.diff(y) < 0.0))

ml_checks = pd.DataFrame({
    "training-label quality": ["short averaging", "long averaging"],
    "RMSE vs hidden truth": [rmse_short, rmse_long],
    "negative predictions": [int(np.sum(pred_short < 0)), int(np.sum(pred_long < 0))],
    "monotonicity violations": [monotonicity_violations(pred_short), monotonicity_violations(pred_long)]
})
display(ml_checks)

### Report prompt 4
Which model is closer to the hidden trend? Does a smooth prediction or low training loss prove that the original DSMC labels were converged? What data-quality metadata should accompany future DSMC training data?

> **Your response:** Write a concise answer here for your own record, and include the final version in the PDF report.
>
> ...

## 5. Save tables used in the report

In [ ]:
noise_df.to_csv("week3_lab1_sampling_noise.csv", index=False)
ml_checks.to_csv("week3_lab1_ml_noise_checks.csv", index=False)
print("Saved week3_lab1_sampling_noise.csv")
print("Saved week3_lab1_ml_noise_checks.csv")

## Final checklist for Lab 1
- [ ] Notebook runs from top to bottom.
- [ ] All plots and tables remain visible.
- [ ] Fitted noise slopes are reported in the PDF.
- [ ] Heat-flux noise is interpreted.
- [ ] The noisy-label ML comparison is discussed.
- [ ] The final written answers appear in the PDF report.

## Article-output contract

<!-- MIE690A article-aligned validation v3 -->

**Role:** Foundational or supporting notebook; see ARTICLE_FIGURE_MAP.md for its evidence dependency.

All manuscript-facing figures must be generated from retained numerical/model outputs through the documented notebook or shared helper, saved under `results/`, and accompanied by machine-readable metrics. Do not redraw curves by eye or substitute a screenshot for a solver-to-reference comparison. The complete ownership table and exact output filenames are in [`ARTICLE_FIGURE_MAP.md`](../../ARTICLE_FIGURE_MAP.md).
